# dCR Compression Study — Phase 6-RUN: Train Vanilla ×3 + Model A (Dice+dCR) ×3

**Read `PROJECT_MASTER.md` §§9, 10, 17 and `docs/kaggle_run_contract.md` before running this.**

Trains 6 runs total: `vanilla` (Dice only) seeds 1/2/3 and `A_dcr` (Dice+dCR) seeds 1/2/3
(master §10: three seeds, logged **separately** — never averaged into one row).

Every hyperparameter — including `curvature_smooth_sigma=2.0` (A06/A30) — is read from
`configs/*.yaml` inside the mounted repo dataset. **Nothing is hardcoded in this notebook.**
Cell 3 proves this on Kaggle itself, before any GPU time is spent, by dry-running the
composed config and printing the resolved value.

Structured so the SAME loop trains B/B′/C later: once Phase 9 (pruning) produces
`checkpoints/channel_masks/ratio_*.json`, extend `RUN_PLAN` in Cell 4 with the B/B′/C
configs and pass `--channel-mask-hash`. Nothing else in this notebook changes.

GPU: select **P100** (or **GPU T4 ×2**, in which case only `cuda:0` is used — the code is
single-device). See `docs/kaggle_run_contract.md` for duration guidance and whether to
split this into two notebook sessions (Vanilla in one, A in another) to stay inside
Kaggle's session time limit.


In [ ]:
import subprocess, sys

print("Python:", sys.version)

import torch
print("torch:", torch.__version__, "| CUDA:", torch.version.cuda,
      "| available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU visible. Set this notebook's Accelerator to GPU "
          "(P100 or T4 x2) in the settings panel, then Restart & Run All.")

# A34: torch/torchvision are NOT reinstalled -- Kaggle's preinstalled, GPU-matched build is
# used as-is (reinstalling risks silently breaking CUDA). Every other requirements-kaggle.txt
# pin is installed explicitly.
OTHER_DEPS = [
    "opencv-python==4.8.1.78", "scikit-image==0.21.0", "scipy==1.11.3",
    "tifffile==2023.7.10", "pyyaml==6.0.1", "pandas==2.1.1",
    "statsmodels==0.14.0", "imagehash==4.3.1", "thop==0.1.1.post2209072238",
]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *OTHER_DEPS], check=True)
print("\ndependencies installed (torch/torchvision left as Kaggle's own build -- A34)")


In [ ]:
from pathlib import Path
import os, sys

# <<< EDIT THIS if your dataset slug differs from what you named it under "Add Data" >>>
REPO_INPUT = Path("/kaggle/input/dcr-compression-study-repo")

assert REPO_INPUT.exists(), (
    f"repo dataset not found at {REPO_INPUT} -- check Add Data > Your Datasets, "
    f"the slug must match this path exactly")

required = [
    "src", "configs", "experiments",
    "data/processed/preprocess_manifest.csv",
    "data/splits/train.csv", "data/splits/test.csv",
    "data/curvature/curvature_bins.csv",
    "requirements-kaggle.txt",
]
print("Checking the mounted dataset contains everything training needs:\n")
missing = []
for rel in required:
    p = REPO_INPUT / rel
    ok = p.exists()
    print(f"  {'OK ' if ok else 'MISSING'}  {rel}")
    if not ok:
        missing.append(rel)
assert not missing, f"missing paths in the mounted dataset: {missing}"

os.chdir(REPO_INPUT)
sys.path.insert(0, str(REPO_INPUT))
print("\ncwd ->", Path.cwd())


In [ ]:
# Drift-guard proof (A30): compose + validate BOTH configs and print the RESOLVED
# curvature_smooth_sigma before any training happens. If this ever printed anything other
# than 2.0 for A_dcr, or failed validation, that would mean the mounted configs/ has drifted
# from what was verified locally -- stop and investigate rather than train on it.
from experiments.multi_seed_train import main as train_main

for cfg_name in ("configs/model_vanilla.yaml", "configs/model_a_dcr.yaml"):
    print(f"\n{'='*74}\ndry-run: {cfg_name}\n{'='*74}")
    rc = train_main(["--model-config", cfg_name, "--seed-index", "0", "--dry-run"])
    assert rc == 0, f"{cfg_name} failed config validation -- fix before spending GPU time"

print("\nBoth configs validated. Confirm above that A_dcr printed "
      "'smooth_sigma=2.0  <- from YAML, not hardcoded' and vanilla printed "
      "'boundary loss     : none'.")


In [ ]:
OUTPUT_ROOT = Path("/kaggle/working")
for sub in ("checkpoints", "logs", "results/tables"):
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)

# master §10: three seeds, logged SEPARATELY. seed-index 0/1/2 map onto
# training_shared.yaml's seeds.model_init_seeds = [1, 2, 3] -- see src/training/seed_manager.py.
RUN_PLAN = [
    ("configs/model_vanilla.yaml", 0),
    ("configs/model_vanilla.yaml", 1),
    ("configs/model_vanilla.yaml", 2),
    ("configs/model_a_dcr.yaml", 0),
    ("configs/model_a_dcr.yaml", 1),
    ("configs/model_a_dcr.yaml", 2),
]
# Later (B/B'/C, after Phase 9 pruning): append e.g.
#   ("configs/model_b_pruned.yaml", 0)
# and pass --channel-mask-hash <hash from checkpoints/channel_masks/ratio_0.5.json> in Cell 6 --
# nothing else in this notebook changes.

print("Run plan (master §10: 3 seeds per model, logged separately):")
for cfg, idx in RUN_PLAN:
    print(f"  {cfg:<32} seed_index={idx}")


In [ ]:
import json, time

# One subprocess PER RUN (not a direct in-process call): guarantees GPU memory is fully
# released between runs, and a crashed run cannot take the other five with it or corrupt
# the notebook kernel's state (docs/kaggle_run_contract.md §4/§5).
LOG_DIR = OUTPUT_ROOT / "_notebook_run_logs"
LOG_DIR.mkdir(exist_ok=True)

results = []
for cfg, idx in RUN_PLAN:
    tag = f"{Path(cfg).stem}_seedidx{idx}"
    print(f"\n{'='*74}\nSTARTING {tag}\n{'='*74}")
    t0 = time.time()
    cmd = [sys.executable, "experiments/multi_seed_train.py",
           "--model-config", cfg, "--seed-index", str(idx),
           "--device", "cuda", "--output-root", str(OUTPUT_ROOT),
           "--executed-on", "kaggle"]
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    log_path = LOG_DIR / f"{tag}.log"
    log_path.write_text(proc.stdout, encoding="utf-8")

    print("\n".join(proc.stdout.splitlines()[-25:]))   # tail, so problems are visible inline
    dur_min = (time.time() - t0) / 60
    ok = proc.returncode == 0
    print(f"\n--- {tag}: {'OK' if ok else 'FAILED -- see ' + str(log_path)} "
          f"({dur_min:.1f} min) ---")
    results.append({"tag": tag, "config": cfg, "seed_index": idx,
                     "returncode": proc.returncode, "duration_min": round(dur_min, 1),
                     "log": str(log_path)})

print("\n\nSUMMARY")
for r in results:
    print(f"  {'OK ' if r['returncode'] == 0 else 'ERR'}  {r['tag']:<28} "
          f"{r['duration_min']:>6.1f} min")
(LOG_DIR / "summary.json").write_text(json.dumps(results, indent=2), encoding="utf-8")

n_failed = sum(1 for r in results if r["returncode"] != 0)
if n_failed:
    print(f"\n*** {n_failed}/{len(results)} run(s) FAILED. Check their .log files under "
          f"{LOG_DIR} before trusting experiment_log.csv. ***")


In [ ]:
import pandas as pd

log_path = OUTPUT_ROOT / "results" / "tables" / "experiment_log.csv"
assert log_path.exists(), "no experiment_log.csv written -- check the run logs in Cell 6"
df = pd.read_csv(log_path)
print(f"experiment_log.csv: {len(df)} row(s)\n")

display_cols = ["run_id", "model", "seed", "status", "best_metric_value",
                 "val_boundary_f1_2px_best", "train_val_gap_final",
                 "train_val_gap_at_best_epoch", "duration_minutes", "curvature_smooth_sigma",
                 "notes"]
display_cols = [c for c in display_cols if c in df.columns]
print(df[display_cols].to_string(index=False))

# A32/A33: the train/val Boundary-F1 gap monitor. Training set is small (~170 images), so a
# flagged run's results should be inspected before being trusted.
warned = df[df["notes"].astype(str).str.contains("WARNING", na=False)]
if len(warned):
    print(f"\n*** {len(warned)} run(s) flagged an overfit-gap WARNING ***")
    for _, r in warned.iterrows():
        print(f"  {r['run_id']}: {r['notes']}")
else:
    print("\nNo train/val Boundary-F1 gap warnings.")

expected = {f"{m}_seed{s}" for m in ("vanilla", "A_dcr") for s in (1, 2, 3)}
got = set(df["run_id"])
missing = expected - got
print(f"\nexpected 6 run_ids -> missing: {missing or 'none'}")


In [ ]:
import shutil

# Exactly the artefacts docs/kaggle_run_contract.md §4 asks Kaggle to produce, bundled for
# download: checkpoints/, logs/, and the experiment_log.csv rows.
DOWNLOAD_DIR = OUTPUT_ROOT / "download_me"
DOWNLOAD_DIR.mkdir(exist_ok=True)
for sub in ("checkpoints", "logs"):
    src = OUTPUT_ROOT / sub
    if src.exists():
        shutil.copytree(src, DOWNLOAD_DIR / sub, dirs_exist_ok=True)
(DOWNLOAD_DIR / "results" / "tables").mkdir(parents=True, exist_ok=True)
shutil.copy(log_path, DOWNLOAD_DIR / "results" / "tables" / "experiment_log.csv")

archive = shutil.make_archive(str(OUTPUT_ROOT / "phase6_run_artifacts"), "zip", DOWNLOAD_DIR)
print(f"Download this from the notebook's Output pane:\n  {archive}\n")
print("Contains:")
for p in sorted(DOWNLOAD_DIR.rglob('*')):
    if p.is_file():
        print(" ", p.relative_to(DOWNLOAD_DIR))
